# Kaggle bulk uploader — iPhone Downloads (one cell)

1. Paste your Kaggle **new API token** into `KAGGLE_API_TOKEN` in the cell below (between quotes). **Never send the token in ChatGPT.**
2. Run the cell. Choose **one ZIP** from your iPhone Downloads folder. It can contain 25 `.ipynb` files.
3. The cell validates notebooks, verifies your Kaggle credentials, then uploads each one **privately** without executing it.

**Security:** A token pasted in a notebook cell is visible to anyone with access to that notebook. Keep this Colab notebook private; after uploading, delete the token from the cell, clear outputs, and avoid sharing saved versions containing it. Use the current Kaggle API **token string**, not the old JSON file. Uploads may update an existing notebook with the same generated slug.

In [ ]:
# ========== ONLY EDIT THIS VALUE ==========
KAGGLE_API_TOKEN = "PASTE_YOUR_KAGGLE_API_TOKEN_HERE"
# ==========================================

KAGGLE_USERNAME = "zeigleritaiml"

import os, sys, json, re, hashlib, subprocess, zipfile
from pathlib import Path

if not KAGGLE_API_TOKEN.strip() or KAGGLE_API_TOKEN == "PASTE_YOUR_KAGGLE_API_TOKEN_HERE":
    raise ValueError("Paste your Kaggle token into KAGGLE_API_TOKEN first.")

# Install CLI, without leaking token in output
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U", "kaggle"], check=True)
from google.colab import files

env = os.environ.copy()
env["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN.strip()
# Do not print the token.
print("Select ONE ZIP archive from iPhone Downloads.")
uploaded = files.upload()
zip_files = [k for k in uploaded if k.lower().endswith(".zip")]
if len(zip_files) != 1:
    raise ValueError("Select exactly one .zip archive.")

books = {}
with zipfile.ZipFile(zip_files[0]) as archive:
    for item in archive.infolist():
        if item.is_dir() or not item.filename.lower().endswith(".ipynb"):
            continue
        name = Path(item.filename).name
        if name in books:
            raise ValueError("Duplicate notebook name: " + name)
        data = archive.read(item)
        content = json.loads(data)
        if content.get("nbformat") != 4 or not isinstance(content.get("cells"), list):
            raise ValueError("Not a valid notebook: " + name)
        books[name] = data
if not books:
    raise ValueError("No .ipynb notebooks found in ZIP")
print(f"Validated {len(books)} Jupyter notebooks.")

# Fail early if Kaggle credentials do not work
check = subprocess.run(["kaggle", "kernels", "list", "--mine"], env=env, text=True, capture_output=True)
if check.returncode:
    print((check.stderr or check.stdout)[-800:])
    raise RuntimeError("Kaggle authentication failed. Check the token variable.")
print("Kaggle login verified.")

# Upload only after authentication succeeds
root = Path("/content/kaggle_private_uploads")
root.mkdir(parents=True, exist_ok=True)
results = []
for index, (filename, data) in enumerate(sorted(books.items()), 1):
    stem = Path(filename).stem
    base = re.sub(r"[^a-z0-9]+", "-", stem.lower()).strip("-")[:48].rstrip("-") or "notebook"
    digest = hashlib.sha256(filename.encode()).hexdigest()[:8]
    slug = f"{base}-{digest}"
    folder = root / slug
    folder.mkdir(exist_ok=True)
    (folder / filename).write_bytes(data)
    metadata = {
        "id": f"{KAGGLE_USERNAME}/{slug}",
        "title": stem.replace("_", " ")[:100],
        "code_file": filename,
        "language": "python",
        "kernel_type": "notebook",
        "is_private": True,
        "enable_gpu": False,
        "enable_internet": False,
        "dataset_sources": [],
        "competition_sources": [],
        "kernel_sources": [],
        "model_sources": []
    }
    (folder / "kernel-metadata.json").write_text(json.dumps(metadata, indent=2))
    response = subprocess.run(["kaggle", "kernels", "push", "-p", str(folder), "--no-run"], env=env, capture_output=True, text=True)
    ok = response.returncode == 0
    results.append((filename, ok, f"https://www.kaggle.com/code/{KAGGLE_USERNAME}/{slug}"))
    print(f"[{index}/{len(books)}] {'SUCCESS' if ok else 'FAILED'}: {filename}")
    if not ok:
        print((response.stderr or response.stdout)[-700:])
print(f"\nFinished: {sum(ok for _,ok,_ in results)}/{len(results)} notebooks uploaded")
for filename, ok, url in results:
    if ok: print(url)
# Credential remains in this code cell until you remove it manually.
